# 🌾 Leaf Disease Detection — Crop Detection Pipeline
### YOLOv8 (5-Crop Detection) + Groq Llama-4-Scout (Zero-Shot Crop Identification)

**Team:** Ashmit Janjirkar, Hitansh Gada, Aryan Gada, Urav Jain
**Guide:** Dr. Monika Mangla — DJSCE, Dept. of Information Technology

**Classes:** `0: sugarcane` · `1: wheat` · `2: maize` · `3: rice` · `4: soybean`

---

This notebook combines two pipelines:
1. **YOLOv8 Object Detection** — trained on the merged 5-crop dataset (bounding-box localisation)
2. **Groq Vision-Language Model** — zero-shot crop identification fallback using `meta-llama/llama-4-scout-17b-16e-instruct`

Runs on free-tier Colab/Kaggle T4 GPU.

---

## 1. Environment Setup

In [ ]:
# Install dependencies
!pip install ultralytics groq pyyaml -q
!pip install roboflow -q   # optional — only needed if downloading dataset directly

import os, yaml, torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

### 1.1 Local Dataset Path
Point this at your merged `dataset/` folder on your own laptop (no Drive mount needed).

In [ ]:
# Set this to the full local path of your merged dataset folder on your laptop
# Windows example: DATASET_ROOT = r"E:\LDD(IPD)\dataset"
# Mac/Linux example: DATASET_ROOT = "/Users/yourname/LDD/dataset"

DATASET_ROOT = r"E:\LDD(IPD)\dataset"   # ← change this to your actual local dataset path

In [ ]:
import os
print("DATASET_ROOT:", DATASET_ROOT)
print("Exists:", os.path.exists(DATASET_ROOT))
print("Contents:", os.listdir(DATASET_ROOT) if os.path.exists(DATASET_ROOT) else "N/A")

## 2. Dataset — Merged 5-Crop YOLO Dataset

Expected structure (already merged via `merge_datasets.py`):

```
dataset/
├── images/
│   ├── train/
│   ├── val/
│   └── test/
├── labels/
│   ├── train/
│   ├── val/
│   └── test/
└── crop_dataset.yaml
```

If you haven't merged your 5 per-crop folders yet, run `merge_datasets.py` locally first,
or download directly from Roboflow below.

In [ ]:
# ── Option A: Download pre-merged dataset from Roboflow (recommended) ──────
# from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")
# project = rf.workspace("s2s3").project("yoyo9")
# dataset = project.version(1).download("yolov8", location=DATASET_ROOT)

# ── Option B: Use already-merged local dataset ──────────────────────────────
assert os.path.exists(DATASET_ROOT), f"Dataset not found at {DATASET_ROOT} — run merge_datasets.py first or use Option A"
print("Dataset found:", DATASET_ROOT)

In [ ]:
# Write / verify the dataset YAML
DATASET_YAML = os.path.join(DATASET_ROOT, "crop_dataset.yaml")

dataset_config = {
    "path": os.path.abspath(DATASET_ROOT),
    "train": "images/train",
    "val":   "images/val",
    "test":  "images/test",
    "nc": 5,
    "names": {
        0: "sugarcane",
        1: "wheat",
        2: "maize",
        3: "rice",
        4: "soybean",
    }
}

with open(DATASET_YAML, "w") as f:
    yaml.dump(dataset_config, f, default_flow_style=False)

print(f"[✓] Dataset YAML ready → {DATASET_YAML}")
print(yaml.dump(dataset_config))

In [ ]:
# Quick sanity check — count images per split
for split in ["train", "val", "test"]:
    img_dir = os.path.join(DATASET_ROOT, "images", split)
    if os.path.exists(img_dir):
        n = len([f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))])
        print(f"  {split:6s}: {n} images")
    else:
        print(f"  {split:6s}: NOT FOUND")

### 2.1 Visualize a few sample images + labels

In [ ]:
import matplotlib.pyplot as plt
import cv2, random

CLASS_NAMES = ["sugarcane", "wheat", "maize", "rice", "soybean"]
COLORS = [(27,158,117), (217,95,2), (117,112,179), (231,41,138), (102,166,30)]

def draw_yolo_labels(img_path, lbl_path):
    img = cv2.imread(img_path)
    if img is None:
        return None
    h, w = img.shape[:2]
    if os.path.exists(lbl_path):
        with open(lbl_path) as f:
            for line in f.readlines():
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cls_id, xc, yc, bw, bh = int(parts[0]), *map(float, parts[1:5])
                x1 = int((xc - bw/2) * w); y1 = int((yc - bh/2) * h)
                x2 = int((xc + bw/2) * w); y2 = int((yc + bh/2) * h)
                color = COLORS[cls_id % len(COLORS)]
                cv2.rectangle(img, (x1,y1), (x2,y2), color, 2)
                cv2.putText(img, CLASS_NAMES[cls_id], (x1, max(y1-6,0)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

train_img_dir = os.path.join(DATASET_ROOT, "images", "train")
train_lbl_dir = os.path.join(DATASET_ROOT, "labels", "train")

if os.path.exists(train_img_dir):
    sample_imgs = random.sample(os.listdir(train_img_dir), min(4, len(os.listdir(train_img_dir))))
    fig, axes = plt.subplots(1, len(sample_imgs), figsize=(18, 5))
    if len(sample_imgs) == 1:
        axes = [axes]
    for ax, fname in zip(axes, sample_imgs):
        img_path = os.path.join(train_img_dir, fname)
        lbl_path = os.path.join(train_lbl_dir, os.path.splitext(fname)[0] + ".txt")
        vis = draw_yolo_labels(img_path, lbl_path)
        if vis is not None:
            ax.imshow(vis)
            ax.set_title(fname, fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No training images found to visualize yet.")

## 3. YOLOv8 Training

Trains on the merged 5-class dataset. Uses `yolov8n` by default for fast iteration on free-tier T4 —
switch to `yolov8s`/`yolov8m` for higher accuracy once the pipeline is validated.

In [ ]:
from ultralytics import YOLO

MODEL_SIZE = "n"          # n | s | m | l | x
EPOCHS     = 30
IMG_SIZE   = 640
BATCH      = 32
DEVICE     = 0 if torch.cuda.is_available() else "cpu"

model = YOLO(f"yolov8{MODEL_SIZE}.pt")
print(f"[✓] Loaded pretrained yolov8{MODEL_SIZE}.pt")

In [ ]:
import shutil, os

LOCAL_DATASET = "/content/dataset"

if not os.path.exists(LOCAL_DATASET):
    print("Copying dataset from Drive to local disk — this may take a few minutes...")
    shutil.copytree(DATASET_ROOT, LOCAL_DATASET)
    print("[✓] Copy complete.")
else:
    print("Local dataset already exists, skipping copy.")

# Point everything at the local copy from here on
DATASET_ROOT = LOCAL_DATASET
DATASET_YAML = os.path.join(DATASET_ROOT, "crop_dataset.yaml")

print("Now training will read from:", DATASET_ROOT)

In [ ]:
results = model.train(
    data      = DATASET_YAML,
    epochs    = EPOCHS,
    imgsz     = IMG_SIZE,
    batch     = BATCH,
    device    = DEVICE,
    project   = "runs/crop_detect",
    name      = "exp1",

    # Augmentation — tuned for field-condition variability
    hsv_h     = 0.015,
    hsv_s     = 0.7,
    hsv_v     = 0.4,
    flipud    = 0.5,
    fliplr    = 0.5,
    mosaic    = 1.0,
    mixup     = 0.1,
    copy_paste= 0.1,
    degrees   = 10.0,
    translate = 0.1,
    scale     = 0.5,
    shear     = 2.0,
)

BEST_WEIGHTS = "runs/crop_detect/exp1/weights/best.pt"
print(f"\n[✓] Training complete. Best weights → {BEST_WEIGHTS}")

In [ ]:
import os
print(os.path.exists("runs"))
print(os.path.exists("runs/crop_detect"))
if os.path.exists("runs/crop_detect"):
    print(os.listdir("runs/crop_detect"))

In [ ]:
model = YOLO("runs/crop_detect/exp1/weights/best.pt")
metrics = model.val(data=DATASET_YAML, split="val")
print(f"mAP@50: {metrics.box.map50:.4f}")

### 3.1 Training Curves

In [ ]:
from IPython.display import Image, display
results_png = "runs/crop_detect/exp1/results.png"
if os.path.exists(results_png):
    display(Image(filename=results_png, width=900))
else:
    print("results.png not found — check training completed successfully")

## 4. Validation — mAP@50, mAP@50-95

In [ ]:
best_model = YOLO(BEST_WEIGHTS)
metrics = best_model.val(data=DATASET_YAML, split="val")

print(f"mAP@50    : {metrics.box.map50:.4f}")
print(f"mAP@50-95 : {metrics.box.map:.4f}")
print(f"Precision : {metrics.box.mp:.4f}")
print(f"Recall    : {metrics.box.mr:.4f}")

# Per-class breakdown
print("\nPer-class mAP@50:")
for i, name in enumerate(CLASS_NAMES):
    if i < len(metrics.box.ap50):
        print(f"  {name:10s}: {metrics.box.ap50[i]:.4f}")

## 5. Inference — Run YOLOv8 on a Test Image

In [ ]:
TEST_IMAGE = os.path.join(DATASET_ROOT, "images", "test")  # or path to a single image

pred_results = best_model.predict(
    source   = TEST_IMAGE,
    conf     = 0.25,
    save     = True,
    save_txt = True,
    project  = "runs/predict",
    name     = "exp1",
)

for r in pred_results[:5]:
    print(f"\n{r.path}")
    for box in r.boxes:
        cls_id = int(box.cls)
        conf   = float(box.conf)
        print(f"  → {CLASS_NAMES[cls_id]:10s}  conf={conf:.2f}")

In [ ]:
# Visualize a predicted sample
pred_dir = "runs/predict/exp1"
if os.path.exists(pred_dir):
    pred_imgs = [f for f in os.listdir(pred_dir) if f.lower().endswith(('.jpg','.png'))]
    if pred_imgs:
        display(Image(filename=os.path.join(pred_dir, pred_imgs[0]), width=600))

## 6. Export to ONNX (for mobile / edge deployment)

In [ ]:
best_model.export(format="onnx", dynamic=True, simplify=True)
onnx_path = BEST_WEIGHTS.replace(".pt", ".onnx")
print(f"[✓] Exported ONNX model → {onnx_path}")

---
## 7. Vision-Language Model — Zero-Shot Crop Identification (Groq)

Complementary module using **Groq's `llama-4-scout-17b-16e-instruct`** for zero-shot crop
identification directly from an image, without any task-specific training.

Useful as:
- A fallback when the YOLOv8 model isn't yet deployed / covers a new crop
- A quick sanity-check against YOLO predictions
- A lightweight verification layer requiring no GPU

**Free tier:** 30 requests/min, 1,000 requests/day, 30,000 TPM — no credit card required.
Get a key at [console.groq.com](https://console.groq.com).

In [ ]:
import getpass

# Paste your key when prompted (input is hidden) — never hardcode it in the notebook
GROQ_API_KEY = getpass.getpass("Enter your GROQ_API_KEY: ")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [ ]:
import base64
from groq import Groq

CLASSES = ["sugarcane", "wheat", "maize", "rice", "soybean"]

SYSTEM_PROMPT = f"""You are an expert agricultural crop identifier.
Given an image, identify which crop is present from ONLY these 5 classes:
{", ".join(CLASSES)}.

Respond in this exact format:
Crop: <crop name>
Confidence: <High / Medium / Low>
Reason: <one line explanation>

If none of the 5 crops are visible, respond with:
Crop: unknown
Confidence: Low
Reason: <explanation>"""

def load_image_base64(image_path: str):
    ext = os.path.splitext(image_path)[-1].lower()
    mime = {".jpg": "image/jpeg", ".jpeg": "image/jpeg",
            ".png": "image/png", ".webp": "image/webp"}.get(ext, "image/jpeg")
    with open(image_path, "rb") as f:
        data = base64.b64encode(f.read()).decode("utf-8")
    return data, mime

def identify_crop(image_path: str) -> dict:
    client = Groq(api_key=os.environ["GROQ_API_KEY"])
    img_data, mime_type = load_image_base64(image_path)

    response = client.chat.completions.create(
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": [
                {"type": "image_url", "image_url": {"url": f"data:{mime_type};base64,{img_data}"}},
                {"type": "text", "text": "Identify the crop in this image."}
            ]}
        ],
        temperature=0.2,
        max_tokens=200,
    )

    raw = response.choices[0].message.content.strip()
    result = {"raw": raw, "crop": "unknown", "confidence": "Low", "reason": ""}
    for line in raw.splitlines():
        if line.lower().startswith("crop:"):
            result["crop"] = line.split(":", 1)[1].strip().lower()
        elif line.lower().startswith("confidence:"):
            result["confidence"] = line.split(":", 1)[1].strip()
        elif line.lower().startswith("reason:"):
            result["reason"] = line.split(":", 1)[1].strip()
    return result

print("[✓] Groq crop identifier ready")

In [ ]:
# Run zero-shot identification on the same test image used for YOLO inference
sample_image = None
test_dir = os.path.join(DATASET_ROOT, "images", "test")
if os.path.exists(test_dir):
    imgs = [f for f in os.listdir(test_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    if imgs:
        sample_image = os.path.join(test_dir, imgs[0])

if sample_image:
    result = identify_crop(sample_image)
    print("── Groq Vision Result ───────────────────")
    print(f"  Image      : {sample_image}")
    print(f"  Crop       : {result['crop'].upper()}")
    print(f"  Confidence : {result['confidence']}")
    print(f"  Reason     : {result['reason']}")
    print("──────────────────────────────────────────")
    display(Image(filename=sample_image, width=400))
else:
    print("No test image found — set `sample_image` manually to try this.")

### 7.1 Compare YOLOv8 vs Groq VLA on the Same Image
Run both models side by side to cross-verify predictions.

In [ ]:
def compare_predictions(image_path):
    print(f"Image: {image_path}\n")

    # YOLOv8 prediction
    yolo_res = best_model.predict(source=image_path, conf=0.25, save=False, verbose=False)
    print("YOLOv8 detections:")
    if yolo_res and len(yolo_res[0].boxes) > 0:
        for box in yolo_res[0].boxes:
            cls_id = int(box.cls)
            conf   = float(box.conf)
            print(f"  → {CLASS_NAMES[cls_id]:10s}  conf={conf:.2f}")
    else:
        print("  → No detections above threshold")

    # Groq VLA prediction
    groq_res = identify_crop(image_path)
    print(f"\nGroq VLA prediction:")
    print(f"  → {groq_res['crop'].upper():10s}  confidence={groq_res['confidence']}")
    print(f"     reason: {groq_res['reason']}")

if sample_image:
    compare_predictions(sample_image)

---
## 8. Summary

| Component | Model | Purpose |
|---|---|---|
| Object Detection | YOLOv8 (`yolov8n`) | Bounding-box localisation of crop/disease regions |
| Zero-Shot Identification | Groq `llama-4-scout-17b-16e-instruct` | Crop verification without training data |
| Export | ONNX (dynamic axes) | Mobile / edge deployment |

**Artifacts produced by this notebook:**
- `runs/crop_detect/exp1/weights/best.pt` — trained YOLOv8 checkpoint
- `runs/crop_detect/exp1/weights/best.onnx` — exported ONNX model
- `runs/predict/exp1/` — annotated inference outputs

**Next steps:** wrap `best.onnx` in the FastAPI backend, integrate both models into the Flutter app,
and add Grad-CAM explainability (see project roadmap).